This code runs through the first 4 steps of the GIBF algorithm as laid out in the mathematical roadmap.

1) **Generate CSDM** — two cells are provided: one using `scipy.signal.csd` (correct but slow) and one manual STFT-based implementation (fast but not yet validated). Run the scipy cell for reliable results.

2) **Select the number of eigenmodes (MDL/AIC)** — automated with Wax & Kailath (1985) information-theoretic criteria. For each of the 513 frequency bins, the MDL criterion searches over a single integer k (model order) to find where the remaining eigenvalues look like noise. This replaces the old hardcoded k=13. Output is `coherent_eigenmodes` shaped `(513, 87, k*)` where k* is the MDL maximum across bins, and `coherent_eigenmodes_mdl` (ragged list) where each entry is `(87, k_f)` for the bin-specific order.

3) **Construct the transfer matrix A** — vibe coded, returns the correct shape. This is possibly the most important cell as it encodes the geometry of the sensors, the resolution of the map, and the field physics.

4) **Ridge regression to create Q_blurry** — retaining MDL-selected eigenmodes makes this significantly faster than the old all-13 approach. Alpha must still be tuned.

**Next steps:** \
5) Iteratively reweighted least squares \
6) Reduce dimensionality of A as we iterate \
7) Repeat until convergence or a set iteration count \
8) Create final source map

In [130]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import signal

# This cell loads the data into an array like this:  sensor_i = [x_i, y_i, z_i], mag_data = [sensor_1, ..., sensor_m ]
# 27 Sensors x 3 time series gives 87 columns. Each time series has 7201 points so there are 7201 rows

np.set_printoptions(precision=2, suppress=True) # no more ugly decimals

folder_path = "/Users/stridersettgast/Downloads/Mag Data" # Copy your path to the data here
folder = Path(folder_path)
text_files = sorted(folder.glob("*.txt"))

all_columns = []

for file_path in text_files:
    # Load the three time series columns
    data = np.loadtxt(file_path, usecols=(7, 8, 9))
    
    # Add each column individually
    all_columns.append(data[:, 0])
    all_columns.append(data[:, 1])
    all_columns.append(data[:, 2])

# Stack all columns
mag_data = np.column_stack(all_columns)

print(f"✓ Loaded {mag_data.shape[0]} rows × {mag_data.shape[1]} columns")
print(f"\nFirst 5 rows (preview):")
print(mag_data[:5, :5])

✓ Loaded 7201 rows × 87 columns

First 5 rows (preview):
[[-13.26  43.12 -20.89  22.75 -72.66]
 [-13.27  43.11 -20.86  22.73 -72.65]
 [-13.3   43.11 -20.9   22.74 -72.63]
 [-13.3   43.07 -20.89  22.73 -72.62]
 [-13.32  43.09 -20.85  22.74 -72.64]]


In [131]:
# Compute CSDM manually by taking the STFT and the outer product using eigensum for efficiency
# This method would be preferable to scipy`s signal.csd() but we need to make sure it is calculating it properly (it is not)

# --- Configuration ---
fs = 1.0        # Sampling frequency
nperseg = 1024  # Segment length
noverlap = nperseg//2  # 50% overlap

# --- Step 1: Compute STFT for all 87 channels at once ---
# mag_data shape: (7201, 87)
# axis=0 computes the STFT along the time dimension for each sensor
frequencies, times, stft_data = signal.stft(
    mag_data, 
    fs=fs, 
    window='hann', 
    nperseg=nperseg, 
    noverlap=noverlap, 
    axis=0,
    detrend='constant',  # Add detrending to match csd()
    boundary=None
)

# Calculate window correction factor
window = signal.get_window('hann', nperseg)
window_correction = np.sum(window**2)  # For power spectra

# Output data shape: (513, 87, 14) 
# (513 frequency bins, 87 sensors, 14 time segments)
print("stft_data shape:", stft_data.shape)                     
print("Number of frequency bins:", frequencies.shape) 
print("Number of time segments:", times.shape) 

# --- Step 2: Compute CSDM using Einstein Summation ---
# 'f_t' represents (frequencies, sensors, time_segments)
# We multiply the sensor vectors and average over the time segments (t)
csdm_manual = np.einsum('fit,fjt->fij', stft_data, np.conj(stft_data), optimize=True)
csdm_manual = csdm_manual / (fs * times.shape[0] * window_correction) 

# --- Final Shape Check ---
print("CSDM shape:", csdm_manual.shape)

stft_data shape: (513, 87, 14)
Number of frequency bins: (513,)
Number of time segments: (14,)
CSDM shape: (513, 87, 87)


In [132]:
# Compute CSDM using scipy`s signal.csd()
# This method is slow but very simple to use

# --- Configuration ---
fs = 1.0  # Sampling frequency
nperseg = 1024  # Segment length
noverlap = nperseg // 2  # 50% overlap is standard

# Reshape arrays to allow NumPy broadcasting to compute all sensor pairs:
# x_input shape becomes: (7201, 87, 1)
# y_input shape becomes: (7201, 1, 87)
x_input = mag_data[:, :, np.newaxis]
y_input = mag_data[:, np.newaxis, :]

# Compute CSDM along the time axis (axis 0)
# SciPy automatically demeans, applies a Hann window, splits into segments, 
# takes the STFT, performs outer products, and takes the expectation (mean)
frequencies, csdm_csd = signal.csd(
    x_input, 
    y_input, 
    fs=fs, 
    window='hann', 
    nperseg=nperseg, 
    noverlap=noverlap, 
    axis=0
)

# --- Final Shape Check ---
print("Number of frequncy bins:", frequencies.shape)  # Output: (513,)
print("CSDM shape:", csdm_csd.shape) # Output: (513, 87, 87)

Number of frequncy bins: (513,)
CSDM shape: (513, 87, 87)


In [ ]:
# Compute the eigenvalues and eigenvectors of the CSDM, vectorized across all frequency bins.
# eigh() exploits the Hermitian structure and returns real eigenvalues in ascending order.

eigenvalues, eigenvectors = np.linalg.eigh(csdm_csd)

# Sort everything into descending order (largest eigenvalue first)
desc_eigenvalues  = eigenvalues[:, ::-1]                           # (513, 87) real
desc_sqrt_eigenvalues = np.sqrt(np.clip(desc_eigenvalues, 0, None))  # (513, 87)
desc_eigenvectors = eigenvectors[:, :, ::-1]                       # (513, 87, 87)

# Scale each eigenvector by its square-root eigenvalue to form eigenmodes
# Shape: (513, 87, 87)
eigenmodes = desc_eigenvectors * desc_sqrt_eigenvalues[:, np.newaxis, :]

print("Eigenvalues shape :", eigenvalues.shape)    # (513, 87)
print("Eigenvectors shape:", eigenvectors.shape)   # (513, 87, 87)
print("Eigenmodes shape  :", eigenmodes.shape)     # (513, 87, 87)
print()
print("Model order selection (MDL/AIC) is done in the next cell.")
print("coherent_eigenmodes will be defined there.")

In [ ]:
# ── Step 2: MDL / AIC Eigenmode Selection ───────────────────────────────────
#
# Problem: which of the non-zero eigenvalues per bin are real signal vs noise?
#
# Key constraint: the STFT-based CSDM has rank ≤ L (number of snapshots = 14),
# so at most L eigenvalues are non-zero. In practice this notebook produces only
# 13 non-zero eigenvalues per bin (due to windowing / detrending). The 14th is a
# structural zero — not noise, just null-space. We must exclude it before running
# MDL or it permanently contaminates the noise model and drives k* to 1.
#
# Solution: detect the number of numerically significant eigenvalues per bin (P_f)
# and run MDL only within that rank-P_f subspace.
#
# Criterion — Wax & Kailath (1985):
#   For each candidate k, test whether the remaining (P_f - k) eigenvalues are
#   equal under a "spherical noise" model. The log-likelihood ratio is:
#
#     LLR(k) = L·(P_f−k)·[ln AM(λ_{k+1}…λ_{P_f}) − ln GM(λ_{k+1}…λ_{P_f})]   ≥ 0
#
#   LLR = 0 iff all remaining eigenvalues are exactly equal (pure noise).
#   Penalising by the free parameters in the k-source model:
#
#     MDL(k) = LLR(k) + ½·k·(2·P_f−k)·ln L    ← conservative, statistically consistent
#     AIC(k) = LLR(k) +   k·(2·P_f−k)          ← lighter penalty, picks more modes

# ── Parameters ───────────────────────────────────────────────────────────────
n_snapshots = stft_data.shape[2]      # L = 14 time segments
M           = desc_eigenvalues.shape[1]  # 87 sensors


def compute_mdl_aic(lam_desc, L):
    """
    MDL and AIC model-order selection for ordered CSDM eigenvalues.

    Parameters
    ----------
    lam_desc : (P,) array  — significant (non-zero) eigenvalues, descending
    L        : int         — number of snapshots used to form the CSDM

    Returns
    -------
    k_mdl : int      — MDL-optimal number of signal eigenmodes
    k_aic : int      — AIC-optimal number of signal eigenmodes
    mdl   : (P-1,)  — MDL criterion values for k = 0 … P-2
    aic   : (P-1,)  — AIC criterion values for k = 0 … P-2
    """
    P   = len(lam_desc)
    lam = np.clip(lam_desc, 1e-30, None)   # guard log(0)

    mdl = np.full(P - 1, np.inf)
    aic = np.full(P - 1, np.inf)

    for k in range(P - 1):               # need ≥ 2 remaining eigenvalues
        noise    = lam[k:]
        n        = len(noise)

        am       = noise.mean()
        gm       = np.exp(np.log(noise).mean())

        llr      = L * n * (np.log(am) - np.log(gm))   # ≥ 0 by AM-GM

        n_params = k * (2 * P - k)       # free parameters for k-source model

        mdl[k]   = llr + 0.5 * n_params * np.log(L)
        aic[k]   = llr + n_params

    return int(np.argmin(mdl)), int(np.argmin(aic)), mdl, aic


# ── Apply per bin, using only the significant (non-zero) eigenvalues ─────────
# Threshold: eigenvalues below 1e-6 × the largest eigenvalue in the bin are
# treated as structural zeros and excluded from the MDL search.
REL_THRESHOLD = 1e-6

k_mdl_all  = np.zeros(513, dtype=int)
k_aic_all  = np.zeros(513, dtype=int)
p_rank_all = np.zeros(513, dtype=int)   # effective rank per bin (for diagnostics)

for f in range(513):
    lam_f     = desc_eigenvalues[f]
    threshold = lam_f[0] * REL_THRESHOLD
    n_sig     = int(np.sum(lam_f > threshold))
    n_sig     = max(n_sig, 2)             # need at least 2 eigenvalues to test
    p_rank_all[f] = n_sig

    k_mdl_all[f], k_aic_all[f], _, _ = compute_mdl_aic(
        lam_f[:n_sig], n_snapshots
    )


# ── Build coherent eigenmode outputs ─────────────────────────────────────────

# Ragged list: coherent_eigenmodes_mdl[f] has shape (87, k_mdl_all[f])
coherent_eigenmodes_mdl = [
    eigenmodes[f, :, : k_mdl_all[f]] for f in range(513)
]

# Fixed-shape array (MDL maximum k) — drop-in for old coherent_eigenmodes[:,:,:13]
k_max_mdl           = int(k_mdl_all.max())
coherent_eigenmodes = eigenmodes[:, :, :k_max_mdl]


# ── Summary ───────────────────────────────────────────────────────────────────
print("=" * 58)
print("  MDL / AIC Eigenmode Selection — Results")
print("=" * 58)
print(f"  Snapshots L            : {n_snapshots}")
print(f"  Sensors   M            : {M}")
print(f"  Effective rank P (per bin) — "
      f"min={p_rank_all.min()}, max={p_rank_all.max()}, mean={p_rank_all.mean():.1f}")
print()
print(f"  MDL k*  — min={k_mdl_all.min():2d}, max={k_mdl_all.max():2d}, "
      f"mean={k_mdl_all.mean():.1f}, median={int(np.median(k_mdl_all))}")
print(f"  AIC k*  — min={k_aic_all.min():2d}, max={k_aic_all.max():2d}, "
      f"mean={k_aic_all.mean():.1f}, median={int(np.median(k_aic_all))}")
print()
print(f"  coherent_eigenmodes        : {coherent_eigenmodes.shape}")
print(f"  coherent_eigenmodes_mdl[0] : {coherent_eigenmodes_mdl[0].shape}")
print()

# ── Single-bin diagnostic ─────────────────────────────────────────────────────
f_test    = 200
lam_test  = desc_eigenvalues[f_test]
n_sig_t   = int(np.sum(lam_test > lam_test[0] * REL_THRESHOLD))
k_f, k_aic_f, mdl_v, aic_v = compute_mdl_aic(lam_test[:n_sig_t], n_snapshots)

print(f"  Bin {f_test} diagnostic  (P_effective = {n_sig_t}):")
print(f"    Significant eigenvalues : {np.round(lam_test[:n_sig_t], 4)}")
print(f"    MDL values (k=0…{n_sig_t-2})   : {np.round(mdl_v, 1)}")
print(f"    AIC values (k=0…{n_sig_t-2})   : {np.round(aic_v, 1)}")
print(f"    MDL k*                  : {k_f}")
print(f"    AIC k*                  : {k_aic_f}")
print(f"    Ragged shape check      : {coherent_eigenmodes_mdl[f_test].shape}  (expect (87, {k_f}))")
print()

# ── Assertions ────────────────────────────────────────────────────────────────
assert coherent_eigenmodes_mdl[f_test].shape == (M, k_f),        "Ragged shape mismatch"
assert coherent_eigenmodes.shape == (513, M, k_max_mdl),          "Fixed-k shape mismatch"
assert np.all((k_mdl_all >= 0) & (k_mdl_all <= p_rank_all - 2)), "k_mdl out of range"
assert np.all(mdl_v >= 0),                                         "MDL must be ≥ 0 (AM-GM)"

print("  All assertions passed. ✓")

In [ ]:
# Scree plot — eigenvalue distribution for a single frequency bin.
# The MDL/AIC cutoffs are now drawn automatically from k_mdl_all / k_aic_all.
# Change bin_index to inspect any of the 513 frequency bins.

bin_index = 200

bin_sqrt_eigs  = desc_sqrt_eigenvalues[bin_index, :n_snapshots]   # top L non-zero
components     = np.arange(1, len(bin_sqrt_eigs) + 1)

variance_explained = (bin_sqrt_eigs / bin_sqrt_eigs.sum()) * 100
cumulative_variance = np.cumsum(variance_explained)

k_mdl_bin = k_mdl_all[bin_index]
k_aic_bin = k_aic_all[bin_index]

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.bar(components, variance_explained, color='steelblue', alpha=0.7, label='Individual variance')
ax1.set_xlabel('Eigenmode index')
ax1.set_ylabel('Variance explained (%)', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.set_title(f'Scree plot — Frequency bin {bin_index}  |  MDL k*={k_mdl_bin}, AIC k*={k_aic_bin}')

ax2 = ax1.twinx()
ax2.plot(components, cumulative_variance, color='darkorange', marker='o',
         linewidth=2, label='Cumulative variance')
ax2.set_ylabel('Cumulative variance (%)', color='darkorange')
ax2.tick_params(axis='y', labelcolor='darkorange')

ax1.axvline(x=k_mdl_bin, color='crimson', linestyle='--', linewidth=1.5,
            label=f'MDL cutoff  k={k_mdl_bin}')
ax1.axvline(x=k_aic_bin, color='purple',  linestyle=':',  linewidth=1.5,
            label=f'AIC cutoff  k={k_aic_bin}')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

fig.tight_layout()
plt.show()

In [135]:
# Transfer matrix configuration object

from typing import Tuple, List
from dataclasses import dataclass

@dataclass
class GridPoint:
    """Represents a point in 3D space"""
    x: float
    y: float
    z: float

class GreenFunctionBasis:
    """Different types of source basis functions"""
    
    MONOPOLE = 0      # Scalar source
    DIPOLE_X = 1      # Dipole oriented in x-direction
    DIPOLE_Y = 2      # Dipole oriented in y-direction
    DIPOLE_Z = 3      # Dipole oriented in z-direction
    # Add more as needed: quadrupole, etc.
    
    @staticmethod
    def count_types():
        return 4  # Monopole + 3 dipole components

class TransferMatrixBuilder:
    def __init__(self, sensor_positions: np.ndarray):
        """
        Parameters:
        sensor_positions: (87, 3) array of (x, y, z) for each sensor
        """
        self.sensor_positions = sensor_positions  # (87, 3)
        self.n_sensors = 87
        
    def greens_monopole(self, sensor_pos: np.ndarray, source_pos: np.ndarray) -> float:
        """
        Green's function for monopole source.
        For magnetic fields: 1/r potential
        """
        r = np.linalg.norm(sensor_pos - source_pos)
        if r < 1e-10:  # Avoid singularity
            return 0
        return 1.0 / r
    
    def greens_dipole(self, sensor_pos: np.ndarray, source_pos: np.ndarray, 
                      orientation: np.ndarray) -> np.ndarray:
        """
        Green's function for dipole source.
        For magnetic fields: gradient of 1/r in orientation direction
        """
        r_vec = sensor_pos - source_pos
        r = np.linalg.norm(r_vec)
        if r < 1e-10:
            return np.zeros(3)
        
        # Dipole field: (3*(m·r̂)r̂ - m) / r^3 for magnetic dipole
        # Or gradient of monopole for scalar potential
        r_hat = r_vec / r
        m = orientation / np.linalg.norm(orientation)
        
        # Dipole field at sensor
        field = (3 * np.dot(m, r_hat) * r_hat - m) / (r**3)
        return field
    
    def compute_transfer_matrix(self, grid_points: List[GridPoint], 
                               basis_types: List[int]) -> np.ndarray:
        """
        Build the full transfer matrix.
        
        Parameters:
        grid_points: K grid points in 3D space
        basis_types: List of N basis function types to use
        
        Returns:
        A: Transfer matrix with shape (87, N*K)
        """
        n_sensors = 87
        K = len(grid_points)
        N = len(basis_types)
        n_cols = K * N
        
        # Initialize transfer matrix
        A = np.zeros((n_sensors, n_cols))
        
        for k, source_pos in enumerate(grid_points):
            source_array = np.array([source_pos.x, source_pos.y, source_pos.z])
            
            for b, basis_type in enumerate(basis_types):
                col_idx = k * N + b
                
                for i, sensor_pos in enumerate(self.sensor_positions):
                    if basis_type == GreenFunctionBasis.MONOPOLE:
                        A[i, col_idx] = self.greens_monopole(sensor_pos, source_array)
                        
                    elif basis_type in [GreenFunctionBasis.DIPOLE_X, 
                                       GreenFunctionBasis.DIPOLE_Y,
                                       GreenFunctionBasis.DIPOLE_Z]:
                        # Orientation vector for dipole
                        orientation = np.zeros(3)
                        if basis_type == GreenFunctionBasis.DIPOLE_X:
                            orientation[0] = 1.0
                        elif basis_type == GreenFunctionBasis.DIPOLE_Y:
                            orientation[1] = 1.0
                        elif basis_type == GreenFunctionBasis.DIPOLE_Z:
                            orientation[2] = 1.0
                            
                        field = self.greens_dipole(sensor_pos, source_array, orientation)
                        # For scalar magnetometers, you might want a specific component
                        # Or use vector field components
                        A[i, col_idx] = field[2]  # Example: vertical component (z)
                        # Or store all three components separately
                        
        return A

In [136]:
# Transfer Matrix Configuration

# Step 1: initialize sensor positions (87 sensors)
# In reality, we would load the actual magnetometer positions
np.random.seed(42)  # For reproducibility

# Example: Randomly distributed in spherical coordinates
# This creates sensors at given latitudes and longitudes

EARTH_RADIUS = 6371000.0

# Choose latitudes and longitudes
latitudes = np.linspace(-80, 80, 10)  # 10 latitudes (skip poles)
longitudes = np.linspace(0, 360, 9, endpoint=False)  # 9 longitudes
# 10 × 9 = 90 points, we'll take first 87

sensor_positions = []

for lat in latitudes:
    for lon in longitudes[:9]:  # Take first 9 longitudes
        # Convert to radians
        lat_rad = np.radians(lat)
        lon_rad = np.radians(lon)
        
        # Spherical to Cartesian
        x = EARTH_RADIUS * np.cos(lat_rad) * np.cos(lon_rad)
        y = EARTH_RADIUS * np.cos(lat_rad) * np.sin(lon_rad)
        z = EARTH_RADIUS * np.sin(lat_rad)
        
        sensor_positions.append([x, y, z])
        
        if len(sensor_positions) >= 87:
            break
    if len(sensor_positions) >= 87:
        break

sensor_positions = np.array(sensor_positions)

# Step 2: Define your grid points (source locations)
# These are potential source locations you want to image
grid_x = np.linspace(-6, 6, 10)   # 10 points in x
grid_y = np.linspace(-6, 6, 10)   # 10 points in y
grid_z = np.linspace(-3, 3, 5)    # 5 points in z

grid_points = []
for x in grid_x:
    for y in grid_y:
        for z in grid_z:
            grid_points.append(GridPoint(x, y, z))

K = len(grid_points)  # Number of grid points

# Step 3: Define which basis functions to use
basis_types = [
    GreenFunctionBasis.MONOPOLE,
    GreenFunctionBasis.DIPOLE_X,
    GreenFunctionBasis.DIPOLE_Y,
    GreenFunctionBasis.DIPOLE_Z
]
N = len(basis_types)  # N = 4 basis functions per grid point

# Step 4: Build the transfer matrix A
builder = TransferMatrixBuilder(sensor_positions)
A = builder.compute_transfer_matrix(grid_points, basis_types)

print(f"A is of shape: {A.shape}")

A is of shape: (87, 2000)


In [137]:
# Ridge regression on each eigenmode

alpha = 0.1 # This parameter can be tuned

# --- Setup Dimensions ---
n_freqs, n_channels, n_modes = coherent_eigenmodes.shape  # (513, 87, 13)

# Define your feature matrix 'A' and regularization parameter 'alpha'
# 'A' must have 87 rows to match the 87 channels in your eigenmodes
# Shape of A: (87, n_features)
n_features = A.shape[1] 

# Initialize an empty array to store the regression weights (x)
# Shape: (513, n_features, 13)
Q_blurry = np.zeros((n_freqs, n_features, n_modes), dtype=complex)

# --- Optimized Execution Loop ---
# Pre-compute normal equation matrices that depend only on 'A' and 'alpha'
ATA = A.T @ A
reg_matrix = ATA + alpha * np.eye(n_features)
L = np.linalg.cholesky(reg_matrix)
LT = L.T

# Loop over each frequency bin
for f in range(n_freqs):
    # Extract all 13 eigenmodes for this specific frequency bin
    # Shape: (87, 13) -> Process all 13 modes at once as a batch matrix 'b'
    b_batch = coherent_eigenmodes[f, :, :]
    
    # Solve L y = A^T b for all 13 columns simultaneously
    ATb = A.T @ b_batch
    y = np.linalg.solve(L, ATb)

    # Solve L^T x = y to get the final regression coefficients
    x_batch = np.linalg.solve(LT, y)
    
    # Store the result for this frequency bin
    Q_blurry[f, :, :] = x_batch

# --- Shape Verification ---
print("Q_blurry shape:", Q_blurry.shape)  # Output: (513, 2000, 13)

Q_blurry shape: (513, 2000, 13)


In [138]:
# Create the blurry source map

# New shape: (513, K, N, 13)
Q_blurry_reshaped = Q_blurry.reshape(513, K, N, 13)

# Sum magnitudes squared across the source types and eigenmodes to isolate the spatial power grid
source_power_blurry = np.sum(np.abs(Q_blurry_reshaped)**2, axis=(2, 3))

# --- Shape Verification ---
print("Q_blurry_reshaped shape:", Q_blurry_reshaped.shape)      # Output: (513, K, N, 13)
print("Source Power Map shape:", source_power_blurry.shape)  # Output: (513, K)

Q_blurry_reshaped shape: (513, 500, 4, 13)
Source Power Map shape: (513, 500)
